In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent

print(f"Project path is: {project_root}")

# Add project root directory to Python path so local modules (src/) can be imported
sys.path.append(str(project_root))

Project path is: /Users/jeet/Documents/Development/Projects/AI-Engineering-Portfolio/pdf-rag-assistant


# 1. PDF Extraction

In [2]:
from src.document_loader import get_pdf_metadata, load_pdf

/Users/jeet/Documents/Development/Projects/AI-Engineering-Portfolio/pdf-rag-assistant/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
pdf_path = './../data/pdfs/Bert_Model.pdf'

documents = load_pdf(pdf_path)
print(f"Total pages: {len(documents)}")

Total pages: 16


### Check first page:

In [4]:
first_page = documents[0]

print(first_page.page_content[:500])
print(first_page.metadata)

Proceedings of NAACL-HLT 2019, pages 4171–4186
Minneapolis, Minnesota, June 2 - June 7, 2019. c⃝2019 Association for Computational Linguistics
4171
BERT: Pre-training of Deep Bidirectional Transformers for
Language Understanding
Jacob Devlin Ming-Wei Chang Kenton Lee Kristina Toutanova
Google AI Language
{jacobdevlin,mingweichang,kentonl,kristout}@google.com
Abstract
We introduce a new language representa-
tion model called BERT, which stands for
Bidirectional Encoder Representations from
Transf
{'producer': 'pdfTeX-1.40.18', 'creator': 'LaTeX with hyperref package', 'creationdate': '2019-04-29T17:36:03+00:00', 'author': 'Jacob Devlin ; Ming-Wei Chang ; Kenton Lee ; Kristina Toutanova', 'keywords': '', 'moddate': '2019-04-29T17:36:03+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.18 (TeX Live 2017) kpathsea version 6.2.3', 'subject': 'N19-1 2019', 'title': 'BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding', 'trapped': '/False', '

### Get metadata:

In [5]:
metadata = get_pdf_metadata(documents)
metadata

{'total_page': 16, 'source': '../data/pdfs/Bert_Model.pdf'}

# 2. Text Chunking

In [6]:
from src.chunking import split_document

In [7]:
chunks = split_document(
	documents,
	chunk_size=1000,
	chunk_overlap=100
)

print(f"Total chunks: {len(chunks)}")

Total chunks: 78


In [8]:
print("First 3 chunks are:\n")
for chunk in chunks[:3]:
	print(f"Chunk {chunk.metadata["chunk_id"]} is:")
	print(chunk)
	print("-" * 50)

First 3 chunks are:

Chunk 1 is:
page_content='Proceedings of NAACL-HLT 2019, pages 4171–4186
Minneapolis, Minnesota, June 2 - June 7, 2019. c⃝2019 Association for Computational Linguistics
4171
BERT: Pre-training of Deep Bidirectional Transformers for
Language Understanding
Jacob Devlin Ming-Wei Chang Kenton Lee Kristina Toutanova
Google AI Language
{jacobdevlin,mingweichang,kentonl,kristout}@google.com
Abstract
We introduce a new language representa-
tion model called BERT, which stands for
Bidirectional Encoder Representations from
Transformers. Unlike recent language repre-
sentation models (Peters et al., 2018a; Rad-
ford et al., 2018), BERT is designed to pre-
train deep bidirectional representations from
unlabeled text by jointly conditioning on both
left and right context in all layers. As a re-
sult, the pre-trained BERT model can be ﬁne-
tuned with just one additional output layer
to create state-of-the-art models for a wide
range of tasks, such as question answering and
lang

# 3. Embeddings

In [9]:
from src.embeddings import generate_embeddings, get_embedding_model

In [19]:
embedding_model = get_embedding_model()

# No need to create embeddings here, as we will do this in create_vector_store from vector_store.py
embeddings = generate_embeddings(
	documents=chunks,
	embedding_model=embedding_model
)

Using device: mps


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8744.18it/s]


# 4. Vector Store

In [20]:
from src.vector_store import (
	create_retriver,
	create_vector_store,
	load_vector_store,
	save_vector_store,
)

### Create FAISS Vector Store

In [21]:
vector_store = create_vector_store(
	documents=chunks,
	embedding_model=embedding_model
)

print("FAISS vector store created successfully.")

FAISS vector store created successfully.


### Save FAISS Index

In [22]:
save_vector_store(vector_store)

print("FAISS index saved successfully.")

FAISS index saved successfully.


### Verify Files

In [23]:
from pathlib import Path

index_path = Path("./vectorstore/faiss_index")

for file in index_path.iterdir():
	print(file.name)

index.faiss
index.pkl


### Load FAISS Index

In [24]:
loaded_vector_store = load_vector_store(
	embedding_model
)

print("FAISS index loaded successfully.")

FAISS index loaded successfully.


### Verify Index Size

In [25]:
loaded_vector_store.index.ntotal

78

### Verify Stored Documents

In [26]:
len(loaded_vector_store.docstore._dict)

78

### Summary

In [27]:
print("=" * 50)
print("FAISS Integration Successful")
print("=" * 50)

print(f"Pages Loaded      : {len(documents)}")
print(f"Chunks Created    : {len(chunks)}")
print(f"Embeddings        : {len(embeddings)}")
print(f"Vectors in FAISS  : {loaded_vector_store.index.ntotal}")

FAISS Integration Successful
Pages Loaded      : 16
Chunks Created    : 78
Embeddings        : 78
Vectors in FAISS  : 78


In [28]:
retrival = create_retriver(
	vector_store=vector_store
)

print("Retrival loaded succesfully")

Retrival loaded succesfully
